# 02 - AutoAugment ResNet (CIFAR-10, <5M params)

**Model:** plain ResNet, blocks [4,4,3], channels [64,128,256]. 4,697,162 params (uses more of the 5M budget).
**Training:** SGD+Nesterov, lr 0.1, cosine 200 epochs, weight decay 5e-4, label smoothing 0.1, batch 128, mixed precision. Augmentation: random crop + flip + **AutoAugment (CIFAR-10 policy)** + random erasing (p=0.5).
**Validation split:** 2% monitor (trains on ~98% of the data).
**Result:** validation accuracy 96.40%, public leaderboard 0.883.

Adding AutoAugment and using nearly all the training data closed part of the gap between validation and the held-out eval set (0.865 -> 0.883), confirming the eval set benefits from stronger regularization.

In [ ]:
import functools; print=functools.partial(print, flush=True)
import os, glob, time, random
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
DEVICE="cuda" if torch.cuda.is_available() else "cpu"; assert DEVICE=="cuda","Turn on GPU."
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

In [ ]:
def _f(f): h=glob.glob(f"/kaggle/input/**/{f}",recursive=True); return h[0] if h else None
def _d(h):
    for dp,dn,fn in os.walk("/kaggle/input"):
        if h in os.path.basename(dp).lower() and any(x.lower().endswith(".png") for x in fn[:50]): return dp
    return None
TRAIN_CSV,TRAIN_DIR=_f("train_labels.csv"),_d("train")
TEST_CSV,TEST_DIR,SAMPLE=_f("test.csv"),_d("test"),_f("sample_submission.csv")
assert all([TRAIN_CSV,TRAIN_DIR,TEST_CSV,TEST_DIR,SAMPLE]),"Data not attached (+ Add Input the competition)."

In [ ]:
MEAN,STD=(0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616)
train_tf=T.Compose([T.RandomCrop(32,padding=4),T.RandomHorizontalFlip(),T.AutoAugment(T.AutoAugmentPolicy.CIFAR10),T.ToTensor(),T.Normalize(MEAN,STD),T.RandomErasing(p=0.5)])
eval_tf=T.Compose([T.ToTensor(),T.Normalize(MEAN,STD)])
def preload(df,d):
    a=np.zeros((len(df),32,32,3),np.uint8)
    for i,fn in enumerate(df["filename"].values): a[i]=np.array(Image.open(os.path.join(d,fn)).convert("RGB"))
    return a
class DS(Dataset):
    def __init__(s,im,me,tf): s.im,s.me,s.tf=im,np.asarray(me),tf
    def __len__(s): return len(s.im)
    def __getitem__(s,i): return s.tf(Image.fromarray(s.im[i])), int(s.me[i])
df=pd.read_csv(TRAIN_CSV); print("preloading train ..."); imgs=preload(df,TRAIN_DIR); lab=df["label"].values
perm=np.random.default_rng(SEED).permutation(len(df)); nv=int(len(df)*0.02); vi,ti=perm[:nv],perm[nv:]
train_loader=DataLoader(DS(imgs[ti],lab[ti],train_tf),batch_size=128,shuffle=True,num_workers=4,pin_memory=True,drop_last=True,persistent_workers=True)
val_loader=DataLoader(DS(imgs[vi],lab[vi],eval_tf),batch_size=512,shuffle=False,num_workers=4,pin_memory=True,persistent_workers=True)
print(f"train={len(ti)}  val={len(vi)}")

In [ ]:
class BasicBlock(nn.Module):
    def __init__(s,ic,oc,k=3,st=1):
        super().__init__(); p=k//2
        s.c1=nn.Conv2d(ic,oc,k,st,p,bias=False); s.b1=nn.BatchNorm2d(oc)
        s.c2=nn.Conv2d(oc,oc,k,1,p,bias=False); s.b2=nn.BatchNorm2d(oc)
        s.sc=nn.Sequential() if (st==1 and ic==oc) else nn.Sequential(nn.Conv2d(ic,oc,1,st,bias=False),nn.BatchNorm2d(oc))
    def forward(s,x): o=F.relu(s.b1(s.c1(x))); o=s.b2(s.c2(o)); return F.relu(o+s.sc(x))
class ResNet(nn.Module):
    def __init__(s,bl,ch,stem=64,k=3,nc=10):
        super().__init__(); s.stem=nn.Sequential(nn.Conv2d(3,stem,3,1,1,bias=False),nn.BatchNorm2d(stem),nn.ReLU(True))
        L,ic=[],stem
        for i,(nb,oc) in enumerate(zip(bl,ch)):
            st=1 if i==0 else 2
            for b in range(nb): L.append(BasicBlock(ic,oc,k,st if b==0 else 1)); ic=oc
        s.layers=nn.Sequential(*L); s.head=nn.Linear(ic,nc)
    def forward(s,x): x=s.stem(x); x=s.layers(x); return s.head(F.adaptive_avg_pool2d(x,1).flatten(1))
cp=lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)

In [ ]:
@torch.no_grad()
def vacc(m):
    m.eval(); c=t=0
    for x,y in val_loader:
        x,y=x.to(DEVICE),y.to(DEVICE)
        with torch.amp.autocast("cuda"): o=m(x)
        c+=(o.argmax(1)==y).sum().item(); t+=y.size(0)
    return c/max(t,1)

In [ ]:
model=ResNet([4,4,3],[64,128,256]).to(DEVICE); n=cp(model); print("params:",f"{n:,}"); assert n<=5_000_000
ce=nn.CrossEntropyLoss(label_smoothing=0.1)
opt=torch.optim.SGD(model.parameters(),lr=0.1,momentum=0.9,weight_decay=5e-4,nesterov=True)
sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=200); scaler=torch.amp.GradScaler("cuda")
best,bs=0.0,None
for ep in range(200):
    model.train(); t0=time.time()
    for x,y in train_loader:
        x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"): loss=ce(model(x),y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    sch.step(); a=vacc(model)
    if a>best: best=a; bs={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    if ep%5==0 or ep==199: print(f"ep {ep:3d}  val {a*100:5.2f}%  best {best*100:5.2f}%  {time.time()-t0:4.1f}s")
model.load_state_dict(bs); torch.save({"state_dict":model.state_dict(),"val_acc":best},"aug_443.pt")
print("best val",f"{best*100:.2f}%")

In [ ]:
# Flip-only test-time augmentation (10-crop was tested and hurt on this eval set)
tdf=pd.read_csv(TEST_CSV); sdf=pd.read_csv(SAMPLE); print("preloading test ..."); tnp=preload(tdf,TEST_DIR)
tl=DataLoader(DS(tnp,tdf["id"].values,eval_tf),batch_size=512,shuffle=False,num_workers=4,pin_memory=True)
model.eval(); ids=[]; pr=[]
with torch.no_grad():
    for x,ix in tl:
        x=x.to(DEVICE)
        with torch.amp.autocast("cuda"): lg=model(x)+model(torch.flip(x,dims=[3]))
        pr.append(lg.argmax(1).cpu().numpy()); ids.append(ix.numpy())
ids=np.concatenate(ids); pr=np.concatenate(pr); pm=dict(zip(ids.tolist(),pr.tolist()))
ic,lc=sdf.columns[0],sdf.columns[1]; out=sdf.copy(); out[lc]=out[ic].map(pm).astype(int)
assert out[lc].notna().all(); out.to_csv("submission.csv",index=False); print("wrote submission.csv"); print(out[lc].value_counts().sort_index())